# Experimento H2 - Catastrophe Analysis - Grupo B
## T7 - Analisis: Wilcoxon pareado

Este notebook **no entrena nada**. Lee los `resultados_<brazo>.txt` que dejaron
los 5 notebooks de brazo y corre el test.

> **H2.** La mejora en la ganancia proviene de senalarle al modelo que el dato
> falta, no de reconstruir su valor.

### Los 4 contrastes, declarados ANTES de correr

| # | contraste | que aisla | H2 predice |
|---|---|---|---|
| 1 | A1 vs A0 | el efecto de senalar, sin reconstruir | **A1 > A0** |
| 2 | A3 vs A1 | reconstruir en vez de senalar | **A3 < A1** |
| 3 | A4 vs A3 | **el flag, con el valor constante** | **A4 > A3** |
| 4 | A4 vs A1 | el valor, con la senal constante | **A4 ~ A1** |

El contraste 3 es el decisivo: A3 y A4 tienen exactamente la misma imputacion
MICE, misma semilla de datos, mismos hiperparametros. Lo unico que los separa es
la columna `ca_reparado`.

### Por que 7 semillas y no 5

El Wilcoxon pareado de signos con n pares tiene un piso combinatorio: el p-value
mas chico posible es `2 / 2^n` a dos colas.

| n | p minimo (2 colas) |
|---|---|
| 5 | 0.0625 -- **nunca llega a 0.05** |
| 6 | 0.03125 |
| 7 | 0.0156 |

Con 5 semillas, aunque los 5 pares fueran en la misma direccion, el test no
puede reportar significancia a dos colas. Por eso son 7.


---

## Antes de empezar: montar el Drive

Este notebook **no baja el dataset ni entrena nada** — sólo lee los
`resultados_<brazo>.txt` que dejaron los 5 brazos. Pero igual necesita ver el
bucket, asi que hay que montar el Drive.

**Las dos celdas que siguen se corren con el runtime en Python 3**
(Runtime -> Change Runtime Type -> Python 3).

Despues hay que **cambiar el runtime a R** para todo el resto.


In [ ]:
# runtime: PYTHON 3
from google.colab import drive
drive.mount('/content/.drive')


In [ ]:
%%shell
# runtime: PYTHON 3
# solo el symlink al bucket, sin descargar nada
mkdir -p "/content/.drive/My Drive/dm"
mkdir -p /content/buckets
ln -sfn "/content/.drive/My Drive/dm"  /content/buckets/b1

ls -la /content/buckets/b1/exp/


---

## Ahora si: cambiar el runtime a **R**

Runtime -> Change Runtime Type -> R. Y seguir desde aca.


In [ ]:
require("data.table")

# carpeta donde estan las subcarpetas WF6300 ... WF6340
ruta_exp <- "/content/buckets/b1/exp"
if (!dir.exists(ruta_exp))
  stop("No existe ", ruta_exp, ".  Corriste las dos celdas de montaje del ",
       "Drive con el runtime en Python 3, antes de cambiar a R?")
setwd(ruta_exp)

PARAM <- list()
PARAM$brazos <- data.table(
  brazo       = c("A0_cero", "A1_na", "A2_interp", "A3_mice", "A4_mice_flag"),
  experimento = c(6300,      6310,    6320,        6330,      6340),
  senala      = c(FALSE,     TRUE,    FALSE,       FALSE,     TRUE),
  reconstruye = c(FALSE,     FALSE,   TRUE,        TRUE,      TRUE)
)

PARAM$semillas <- c(100043, 200063, 300089, 500069, 700021, 181219, 410341)

PARAM$brazos


### Levantar los resultados de los 5 brazos


In [ ]:
tb <- rbindlist(lapply(seq_len(nrow(PARAM$brazos)), function(i) {
  b <- PARAM$brazos[i]
  f <- file.path(paste0("WF", b$experimento), paste0("resultados_", b$brazo, ".txt"))
  if (!file.exists(f)) {
    warning("FALTA: ", f)
    return(NULL)
  }
  fread(f)
}))

# control: 5 brazos x 7 semillas = 35 filas, sin faltantes ni duplicados
cat("filas leidas:", nrow(tb), " (esperadas: 35)\n")
print(tb[, .N, by = brazo])

faltan <- CJ(brazo = PARAM$brazos$brazo, semilla = PARAM$semillas)[
            !tb, on = .(brazo, semilla)]
if (nrow(faltan)) { cat("\nCORRIDAS QUE FALTAN:\n"); print(faltan) }

tb


### La matriz brazo x semilla


In [ ]:
mat <- dcast(tb, brazo ~ semilla, value.var = "ganancia_suavizada_max")
setcolorder(mat, c("brazo", as.character(PARAM$semillas)))
mat <- mat[match(PARAM$brazos$brazo, brazo)]
print(mat)

cat("\nresumen por brazo:\n")
resumen <- tb[, .(media  = mean(ganancia_suavizada_max),
                  mediana = median(ganancia_suavizada_max),
                  sd     = sd(ganancia_suavizada_max),
                  min    = min(ganancia_suavizada_max),
                  max    = max(ganancia_suavizada_max),
                  envios_medianos = median(envios)),
              by = brazo]
resumen <- resumen[match(PARAM$brazos$brazo, brazo)]
print(resumen)

# el ruido entre semillas DENTRO de un brazo es la vara: si la diferencia
# entre brazos es menor que esto, no hay nada que discutir
cat("\ndesvio entre semillas (el ruido de fondo):",
    round(mean(resumen$sd), 4), "\n")
cat("rango entre las medias de los brazos:",
    round(max(resumen$media) - min(resumen$media), 4), "\n")


### Los 4 contrastes

`wilcox.test(..., paired = TRUE)` sobre las 7 diferencias pareadas por semilla.
Dos colas, y correccion de Holm por los 4 contrastes.


In [ ]:
g <- function(b) tb[brazo == b][match(PARAM$semillas, semilla), ganancia_suavizada_max]

contrastes <- list(
  list(id = 1, a = "A1_na",        b = "A0_cero",   aisla = "senalar, sin reconstruir",     predice = "A1 > A0"),
  list(id = 2, a = "A3_mice",      b = "A1_na",     aisla = "reconstruir en vez de senalar", predice = "A3 < A1"),
  list(id = 3, a = "A4_mice_flag", b = "A3_mice",   aisla = "EL FLAG (valor constante)",     predice = "A4 > A3"),
  list(id = 4, a = "A4_mice_flag", b = "A1_na",     aisla = "el valor (senal constante)",    predice = "A4 ~ A1")
)

res <- rbindlist(lapply(contrastes, function(k) {
  x <- g(k$a); y <- g(k$b)
  w <- suppressWarnings(wilcox.test(x, y, paired = TRUE, exact = TRUE))
  data.table(
    n            = k$id,
    contraste    = paste(k$a, "vs", k$b),
    aisla        = k$aisla,
    H2_predice   = k$predice,
    dif_mediana  = median(x - y),
    a_gana_en    = sum(x > y),
    de           = length(x),
    p_value      = w$p.value
  )
}))

res[, p_holm := p.adjust(p_value, method = "holm")]
res[, significativo := p_holm < 0.05]

print(res)

cat("\npiso del test con", length(PARAM$semillas), "semillas:",
    2 / 2^length(PARAM$semillas), "\n")


### Las diferencias pareadas, semilla por semilla


In [ ]:
for (k in contrastes) {
  x <- g(k$a); y <- g(k$b)
  cat("\n--- ", k$a, " vs ", k$b, "  (", k$aisla, ")\n", sep = "")
  print(data.table(semilla = PARAM$semillas,
                   a = round(x, 4), b = round(y, 4), dif = round(x - y, 4)))
  cat("  a gana en ", sum(x > y), " de ", length(x), " semillas\n", sep = "")
}


### La prediccion secundaria registrada

Registrada el **2026-09-07**, con A0 y A1 ya corridos y **A2, A3 y A4 todavia sin
correr**:

> Los brazos que **senalan** la ausencia (A1, A4) tendran **menor varianza entre
> semillas** que los brazos que no senalan (A0, A2, A3).

Se pone a prueba con el test de **Pitman-Morgan**, que es el correcto para
comparar varianzas de muestras **pareadas** (un `var.test` comun supondria
independencia, y aca los dos brazos comparten las mismas 7 semillas).

El test mas limpio es **A4 vs A3**: comparten imputacion MICE, semilla de datos e
hiperparametros, y solo los separa la columna `ca_reparado`.


In [ ]:
# matriz semillas x brazos
M <- sapply(PARAM$brazos$brazo, g)
rownames(M) <- PARAM$semillas
stopifnot(!anyNA(M))

# Pitman-Morgan: correlacion entre la suma y la diferencia
pitman <- function(x, y) {
  ct <- cor.test(x + y, x - y)
  list(p = ct$p.value, r = unname(ct$estimate))
}

ref <- "A1_na"
tb_var <- rbindlist(lapply(PARAM$brazos$brazo, function(b) {
  pm <- if (b == ref) list(p = NA_real_, r = NA_real_) else pitman(M[, b], M[, ref])
  data.table(brazo = b,
             senala = PARAM$brazos[brazo == b, senala],
             desvio = sd(M[, b]),
             varianza = var(M[, b]),
             razon_vs_A1 = var(M[, b]) / var(M[, ref]),
             p_pitman_vs_A1 = pm$p)
}))
print(tb_var)

cat("\n--- EL TEST DECISIVO: A4 vs A3 (solo los separa ca_reparado) ---\n")
pm43 <- pitman(M[, "A4_mice_flag"], M[, "A3_mice"])
cat(sprintf("  desvio A3 = %.4f    desvio A4 = %.4f\n",
            sd(M[, "A3_mice"]), sd(M[, "A4_mice_flag"])))
cat(sprintf("  razon de varianzas A4/A3 = %.2f\n",
            var(M[, "A4_mice_flag"]) / var(M[, "A3_mice"])))
cat(sprintf("  Pitman-Morgan  p = %.4f\n", pm43$p))
cat("\n  La prediccion se sostiene solo si A4 (senala) es MAS estable que A3.\n")


### Correccion por multiplicidad

Las comparaciones de a pares contra A1 estan **sesgadas**: A1 se eligio como
referencia *despues* de ver que era el brazo mas estable, entre cinco.

La pregunta honesta no es «A1 es mas estable que A2?» sino **«que tan raro es que
*alguno* de los 5 brazos salga tan estable como el que salio?»**

El test de permutacion responde eso y respeta el apareamiento: permuta las
etiquetas de brazo **dentro de cada semilla**, asi que conserva el efecto de la
semilla y destruye solo el efecto del tratamiento.


In [ ]:
set.seed(1)
B <- 20000

obs_min   <- min(apply(M, 2, sd))
obs_razon <- max(apply(M, 2, var)) / min(apply(M, 2, var))

nulo <- replicate(B, {
  P <- t(apply(M, 1, sample))          # permuta los 5 brazos en cada semilla
  c(min(apply(P, 2, sd)),
    max(apply(P, 2, var)) / min(apply(P, 2, var)))
})

p_min   <- mean(nulo[1, ] <= obs_min)
p_razon <- mean(nulo[2, ] >= obs_razon)

cat("=== TEST DE PERMUTACION (", B, "repeticiones ) ===\n")
cat(sprintf("  desvio minimo observado             = %.4f\n", obs_min))
cat(sprintf("  desvio minimo bajo permutacion      = %.4f (mediana)\n", median(nulo[1, ])))
cat(sprintf("  p = P(minimo del nulo <= observado) = %.4f\n\n", p_min))
cat(sprintf("  razon max/min de varianzas observada = %.2f\n", obs_razon))
cat(sprintf("  mediana bajo permutacion             = %.2f\n", median(nulo[2, ])))
cat(sprintf("  p = P(razon del nulo >= observada)   = %.4f\n", p_razon))

cat("\n=== TESTS CLASICOS DE HOMOGENEIDAD (5 grupos) ===\n")
largo <- data.frame(g = rep(colnames(M), each = nrow(M)), v = as.vector(M))
bt <- bartlett.test(v ~ g, data = largo)
fk <- fligner.test(v ~ g, data = largo)
print(bt); print(fk)

tb_perm <- data.table(
  test = c("permutacion: desvio minimo", "permutacion: razon max/min",
           "Bartlett", "Fligner-Killeen"),
  estadistico = c(obs_min, obs_razon, unname(bt$statistic), unname(fk$statistic)),
  p = c(p_min, p_razon, bt$p.value, fk$p.value))
print(tb_perm)


### Potencia - por que el nulo es informativo

Un `p` alto no dice «no hay efecto». Dice «no lo detectamos». Lo que convierte eso
en un resultado es calcular **que tamano de efecto habria podido detectar este
diseno**, con el ruido que realmente se observo.


In [ ]:
d1 <- g("A1_na") - g("A0_cero")   # el contraste 1

cat("=== RUIDO OBSERVADO ===\n")
cat(sprintf("  desvio de las diferencias pareadas A1-A0 = %.4f\n", sd(d1)))
cat(sprintf("  desvio promedio SIN aparear              = %.4f\n",
            mean(c(sd(g("A1_na")), sd(g("A0_cero"))))))
cat("  (si aparear ayudara, el PRIMERO seria menor que el segundo)\n")
cat(sprintf("  correlacion entre brazos = %+.4f\n", cor(g("A1_na"), g("A0_cero"))))

cat("\n=== POTENCIA para detectar la diferencia observada ===\n")
tb_pot <- rbindlist(lapply(c(7, 10, 15, 20, 30, 50, 100), function(n) {
  data.table(n = n,
             potencia = power.t.test(n = n, delta = mean(d1), sd = sd(d1),
                                     sig.level = 0.05, type = "paired")$power)
}))
print(tb_pot)
cat(sprintf("\n  (para una diferencia real de %.4f con desvio pareado %.4f)\n",
            mean(d1), sd(d1)))
cat("  El techo del profesor son 15 semillas.\n")


### Grafico


In [ ]:
pdf("H2_resultados.pdf", width = 9, height = 6)

orden <- PARAM$brazos$brazo
tb[, brazo_f := factor(brazo, levels = orden)]

boxplot(ganancia_suavizada_max ~ brazo_f, data = tb,
        main = "H2 - ganancia por brazo (7 semillas)",
        xlab = "", ylab = "ganancia_suavizada_max",
        col = c("gray85", "steelblue", "gray85", "gray85", "indianred"),
        las = 2, cex.axis = 0.8)

# cada semilla como una linea: se ve si el orden de los brazos es consistente
for (s in PARAM$semillas) {
  y <- tb[semilla == s][match(orden, brazo), ganancia_suavizada_max]
  lines(seq_along(orden), y, col = rgb(0, 0, 0, 0.25), lty = 3)
  points(seq_along(orden), y, pch = 20, col = rgb(0, 0, 0, 0.5), cex = 0.7)
}

dev.off()
cat("grabado H2_resultados.pdf\n")


### Salida para las diapositivas


In [ ]:
fwrite(tb,      file = "H2_todas_las_corridas.txt", sep = "\t")
fwrite(mat,     file = "H2_matriz_brazo_semilla.txt", sep = "\t")
fwrite(resumen, file = "H2_resumen_por_brazo.txt", sep = "\t")
fwrite(res,     file = "H2_wilcoxon.txt", sep = "\t")
fwrite(tb_var,  file = "H2_varianzas.txt", sep = "\t")
fwrite(tb_perm, file = "H2_permutacion.txt", sep = "\t")
fwrite(tb_pot,  file = "H2_potencia.txt", sep = "\t")

cat("\n=========================================================\n")
cat(" CONCLUSION\n")
cat("=========================================================\n")
for (i in seq_len(nrow(res))) {
  r <- res[i]
  cat(sprintf("%d. %-28s dif=%+8.4f  p=%.4f  p_holm=%.4f  %s\n",
      r$n, r$contraste, r$dif_mediana, r$p_value, r$p_holm,
      ifelse(r$significativo, "SIGNIFICATIVO", "no significativo")))
}
cat("\nH2 se sostiene si el contraste 3 (A4 vs A3) da a favor de A4.\n")

cat("\n--- LA PREDICCION SECUNDARIA (varianza) ---\n")
cat(sprintf("  razon de varianzas A4/A3 = %.2f   p = %.4f\n",
    var(M[,"A4_mice_flag"])/var(M[,"A3_mice"]), pm43$p))
cat("  Se sostiene solo si A4 es MAS estable que A3.\n")
cat(sprintf("\n  Corregido por multiplicidad (permutacion): p = %.4f\n", p_min))
cat("  Si ese p no es chico, la estabilidad de A1 no es notable.\n")


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
